### Colab Mount

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab
    from google.colab import drive

    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    src_dir = Path("/content/drive/MyDrive/Research/AI-In-Science-Lab/local_learning/src")
else:
    cwd = Path.cwd().resolve()
    src_dir = cwd if (cwd / "models").is_dir() else cwd / "local_learning" / "src"

os.chdir(src_dir)
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

### Setup

In [ ]:
from typing import Any

import torch
import wandb
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from datasets import get_dataset
from losses import get_loss
from models import get_model

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device] {DEVICE}")

### Config

In [ ]:
DEFAULT_CONFIG = {
    "project": "local-learning",
    "dataset": "cifar10",
    "preprocessing": "none",
    "architecture_type": "cnn1",
    "training_mode": "classification",
    "loss_type": "regular",
    "inference_mode": False,
    "learning_rate": 1e-3,
    "epochs": 3,
    "batch_size": 64,
    "num_workers": 2,
    "lambda_strength": 0.0,
    "kernel_size": 3,
    "correlation_mode": "max",
    "comparison_mode": "shift",
    "normalization_mode": "relu",
    "postcomp_mode": "squared",
    "correlation_loss": "sum",
    "local": False,
    "hidden_ch": 64,
    "k_spatial": 0.2,
    "k_population": None,
    "k_lifetime": 0.2,
    "wta_eval_multiplier": 1.0,
    "log_every_steps": 100,
    "max_eval_batches": 25,
}

### Dataset

In [ ]:
def get_loaders(cfg: dict[str, Any]) -> tuple[DataLoader, DataLoader]:
    train_ds = get_dataset(
        train=True,
        dataset=cfg["dataset"],
        preprocessing=cfg["preprocessing"],
    )
    test_ds = get_dataset(
        train=False,
        dataset=cfg["dataset"],
        preprocessing=cfg["preprocessing"],
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=int(cfg["batch_size"]),
        shuffle=True,
        num_workers=int(cfg["num_workers"]),
        pin_memory=(DEVICE.type == "cuda"),
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=int(cfg["batch_size"]),
        shuffle=False,
        num_workers=int(cfg["num_workers"]),
        pin_memory=(DEVICE.type == "cuda"),
    )
    return train_loader, test_loader

### Architectures

In [ ]:
ARCHITECTURES = [
    "cnn1",
    "cnn2",
    "resnet18",
    "pretrained_resnet18",
    "wta_conv_ae",
]

### Losses

In [ ]:
TRAINING_MODES = ["classification", "reconstruction"]
LOSS_TYPES = ["regular", "redundancy"]

### Training

In [ ]:
def get_config_name(cfg: dict[str, Any]) -> str:
    return (
        f"{cfg['training_mode']}"
        f"-{cfg['architecture_type']}"
        f"-{cfg['dataset']}"
        f"-{cfg['preprocessing']}"
        f"-{cfg['loss_type']}"
        f"-lam_{cfg['lambda_strength']}"
        f"-lr_{cfg['learning_rate']}"
    )


def _capitalize_top_folder(key: str) -> str:
    if "/" not in key:
        return key
    top, rest = key.split("/", 1)
    if not top:
        return key
    return f"{top[:1].upper()}{top[1:]}" + f"/{rest}"


def wandb_log(payload: dict[str, object], *, step: int | None = None) -> None:
    payload = {_capitalize_top_folder(key): value for key, value in payload.items()}
    if step is None:
        wandb.log(payload)
        return
    wandb.log(payload, step=step)


def _loss_value(
    cfg: dict[str, Any],
    criterion,
    output: torch.Tensor,
    xb: torch.Tensor,
    yb: torch.Tensor,
    feature_maps: list,
) -> torch.Tensor:
    if cfg["training_mode"] == "classification":
        return criterion(y=yb, y_pred=output, fm_list=feature_maps)
    if output.shape != xb.shape:
        xb = xb.view_as(output)
    return criterion(reconstruction=output, target=xb, feature_maps=feature_maps)


def _metric_payload(cfg: dict[str, Any], criterion, output: torch.Tensor, yb: torch.Tensor) -> dict[str, float]:
    if cfg["training_mode"] == "classification":
        acc = (output.argmax(dim=1) == yb).to(torch.float32).mean()
        return {
            "Losses/ce": float(getattr(criterion, "last_ce_loss", torch.tensor(0.0)).detach().cpu()),
            "General/acc": float(acc.detach().cpu()),
        }
    return {
        "Train/mse": float(getattr(criterion, "last_mse_loss", torch.tensor(0.0)).detach().cpu()),
    }


def _log_loss_parts(criterion, payload: dict[str, float | int]) -> None:
    payload["Losses/correlation_total"] = float(
        getattr(criterion, "last_corr_total", torch.tensor(0.0)).detach().cpu()
    )
    for layer_name, layer_loss in getattr(criterion, "last_corr_by_layer", {}).items():
        payload[f"Losses/{layer_name}"] = float(layer_loss.detach().cpu())


def train(config: dict[str, Any]) -> dict[str, float]:
    run = wandb.init(project=config["project"], config=config)
    cfg = dict(wandb.config)
    run.name = get_config_name(cfg)

    train_loader, test_loader = get_loaders(cfg)
    cfg["dataset_size"] = len(train_loader.dataset)

    model = get_model(cfg).to(DEVICE)
    criterion = get_loss(cfg).to(DEVICE)
    optimizer = torch.optim.Adam(
        [param for param in model.parameters() if param.requires_grad],
        lr=float(cfg["learning_rate"]),
    )

    def eval_model(epoch: int) -> dict[str, float]:
        model.eval()
        loss_sum = 0.0
        acc_sum = 0.0
        mse_sum = 0.0
        n = 0
        with torch.no_grad():
            for batch_idx, (xb, yb) in enumerate(test_loader):
                if batch_idx >= int(cfg["max_eval_batches"]):
                    break
                xb = xb.to(DEVICE, non_blocking=True)
                yb = yb.to(DEVICE, non_blocking=True)
                if cfg["training_mode"] == "classification":
                    output, feature_maps = model(xb)
                else:
                    output, feature_maps = model(xb, epoch=epoch, inputs_processed_in_epoch=0)
                loss = _loss_value(cfg, criterion, output, xb, yb, feature_maps)
                batch_size = int(xb.shape[0])
                loss_sum += float(loss.detach().cpu()) * batch_size
                if cfg["training_mode"] == "classification":
                    acc = (output.argmax(dim=1) == yb).to(torch.float32).mean()
                    acc_sum += float(acc.detach().cpu()) * batch_size
                else:
                    mse_sum += float(criterion.last_mse_loss.detach().cpu()) * batch_size
                n += batch_size
        model.train()
        metrics = {"Test/loss": loss_sum / max(1, n)}
        if cfg["training_mode"] == "classification":
            metrics["Test/acc"] = acc_sum / max(1, n)
        else:
            metrics["Test/mse"] = mse_sum / max(1, n)
        return metrics

    global_step = 0
    log_every_steps = int(cfg["log_every_steps"])
    eval_interval_steps = max(1, len(train_loader) // 4)

    for epoch in tqdm(range(int(cfg["epochs"])), desc="Epochs"):
        model.train()
        inputs_processed_in_epoch = 0

        for step_in_epoch, (xb, yb) in enumerate(train_loader):
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            if cfg["training_mode"] == "classification":
                output, feature_maps = model(xb)
            else:
                output, feature_maps = model(
                    xb,
                    epoch=epoch,
                    inputs_processed_in_epoch=inputs_processed_in_epoch,
                )

            loss = _loss_value(cfg, criterion, output, xb, yb, feature_maps)
            loss.backward()
            optimizer.step()

            batch_size = int(xb.shape[0])
            inputs_processed_in_epoch += batch_size

            if global_step % log_every_steps == 0:
                log_payload: dict[str, float | int] = {
                    "Train/loss": float(loss.detach().cpu()),
                    "General/epoch": epoch,
                    "General/step": global_step,
                }
                log_payload.update(_metric_payload(cfg, criterion, output, yb))
                if hasattr(model, "last_k"):
                    log_payload["General/last_k"] = int(model.last_k)
                _log_loss_parts(criterion, log_payload)
                wandb_log(log_payload, step=global_step)

            global_step += 1

            if (step_in_epoch + 1) % eval_interval_steps == 0:
                metrics = eval_model(epoch)
                metrics.update({"General/epoch": epoch, "General/step": global_step})
                wandb_log(metrics, step=global_step)

    metrics = eval_model(int(cfg["epochs"]) - 1)
    wandb.finish()
    return metrics

### Run

In [ ]:
# metrics = train(DEFAULT_CONFIG)
# metrics